In [ ]:
# notebooks/exploration.ipynb
import yaml
import numpy as np
from nuscenes.nuscenes import NuScenes
import os
import sys

notebook_path = os.path.abspath(os.getcwd())
project_root = os.path.dirname(notebook_path)
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added project root to sys.path: {project_root}")

# Enable autoreload
%load_ext autoreload
%autoreload 2



from src.core import MDetector, OcclusionResult # MDetector now manages library
from src.data_utils.nuscenes_helper import get_lidar_sweep_data


In [ ]:


# --- Load Config and Initialize NuScenes (as before) ---
config_file_path = '../config/m_detector_config.yaml'
with open(config_file_path, 'r') as f:
    config = yaml.safe_load(f)

nusc = NuScenes(version=config['nuscenes']['version'],
                dataroot=config['nuscenes']['dataroot'],
                verbose=False) # verbose=False to reduce nusc loading output

# --- Initialize MDetector ---
m_detector = MDetector(config=config)

# --- Get a sequence of LiDAR tokens to process ---
scene_token = nusc.scene[1]['token'] # Example: first scene
current_sample_token = nusc.get('scene', scene_token)['first_sample_token']
lidar_sensor_name = config['nuscenes']['lidar_sensor_name']

sweeps_processed = 0
# Process enough sweeps to pass initialization + a few more for testing
start_sweep = 49
max_sweeps_to_process = config['initialization']['num_initial_sweeps_for_map'] + 3 

print(f"Starting M-Detector simulation for {max_sweeps_to_process} sweeps...")
while current_sample_token and sweeps_processed < max_sweeps_to_process + start_sweep:
    if sweeps_processed < start_sweep:
        sweeps_processed += 1
        continue
    sample = nusc.get('sample', current_sample_token)
    lidar_token = sample['data'][lidar_sensor_name]

    print(f"\n--- Processing Sweep {sweeps_processed + 1}/{start_sweep + max_sweeps_to_process} (token: {lidar_token}) ---")
    points_lidar_frame, T_global_lidar, lidar_timestamp = get_lidar_sweep_data(nusc, lidar_token)

    # MDetector creates and adds the DepthImage to its internal library
    current_di = m_detector.add_sweep_and_create_depth_image(
        points_lidar_frame, T_global_lidar, lidar_timestamp
    )

    if m_detector.is_ready_for_processing():
        print("M-Detector is ready. Occlusion checks can be performed against historical DIs.")
        # For now, we just populate. Testing will be in the next cell.
    else:
        print(f"M-Detector initializing: {len(m_detector.depth_image_library)}/{m_detector.min_sweeps_for_processing} DIs collected.")

    if sample['next']:
        current_sample_token = sample['next']
    else:
        break # End of scene
    sweeps_processed += 1

print("\n--- Simulation complete. MDetector library populated. ---")


In [ ]:
# import k3d # Make sure k3d is imported

# if not m_detector.is_ready_for_processing() or len(m_detector.depth_image_library) < 2:
#     print("M-Detector not ready or not enough historical images for occlusion testing.")
#     print(f"Library size: {len(m_detector.depth_image_library)}, Min required for processing: {m_detector.min_sweeps_for_processing}")
# else:
#     print("--- Starting Occlusion Check Testing ---")
    
#     # Get the latest DI (this would be the "current" scan in a real scenario)
#     # Its points are what we'll test FOR occlusion against historical data.
#     # For this test, we'll pick points from this latest DI.
#     latest_di = m_detector.depth_image_library.get_latest_image()
    
#     # Get a historical DI to compare against (e.g., the one before the latest)
#     historical_di = m_detector.depth_image_library.get_image_by_index(-2) 

#     if not latest_di or not historical_di:
#         print("Error: Could not retrieve latest or historical DI for testing.")
#     else:
#         print(f"Testing points from 'Latest DI' (Timestamp: {latest_di.timestamp})")
#         print(f"Against 'Historical DI' (Timestamp: {historical_di.timestamp})")

#         # Select a few points from the 'latest_di' to test
#         points_to_test_global = []
#         points_to_test_sph_in_latest_di_frame = [] # Store their original sph_coords for context

#         # Iterate through pixels of latest_di to find some sample points
#         # We'll take up to `num_points_for_viz_test` points
#         num_points_for_viz_test = 3 
#         for v_idx in range(latest_di.num_pixels_v):
#             if len(points_to_test_global) >= num_points_for_viz_test: break
#             for h_idx in range(latest_di.num_pixels_h):
#                 if len(points_to_test_global) >= num_points_for_viz_test: break
#                 pixel_content = latest_di.pixels[v_idx, h_idx]
#                 if pixel_content and pixel_content['points']:
#                     # Take the first point in this pixel that has a reasonable depth
#                     for pt_info in pixel_content['points']:
#                         if pt_info['sph_coords'][2] > 1.0: # Ensure it's not too close
#                             points_to_test_global.append(pt_info['global_pt'])
#                             points_to_test_sph_in_latest_di_frame.append(pt_info['sph_coords'])
#                             break # Move to next pixel
        
#         if not points_to_test_global:
#             print("Could not find suitable points in 'latest_di' to test.")
        
#         for i, current_pt_g_to_test in enumerate(points_to_test_global):
#             original_sph_coords = points_to_test_sph_in_latest_di_frame[i]
#             print(f"\n--- Test Point {i+1}/{len(points_to_test_global)} ---")
#             print(f"Global Coords: {current_pt_g_to_test}")
#             print(f"Original Spherical (in its own DI frame latest_di): phi={np.rad2deg(original_sph_coords[0]):.1f}°, theta={np.rad2deg(original_sph_coords[1]):.1f}°, d={original_sph_coords[2]:.2f}m")

#             # 1. Pixel-Level Occlusion Check
#             pixel_res, pix_idx_hist, sph_curr_in_hist = \
#                 m_detector.check_occlusion_pixel_level(current_pt_g_to_test, historical_di)
#             print(f"  Pixel-Level Result: {pixel_res.name}")
#             if pix_idx_hist:
#                 print(f"    Projected to Historical DI Pixel: {pix_idx_hist}, Spherical (in hist_di frame): phi={np.rad2deg(sph_curr_in_hist[0]):.1f}°, theta={np.rad2deg(sph_curr_in_hist[1]):.1f}°, d={sph_curr_in_hist[2]:.2f}m")

#             # 2. Point-Level Occlusion Check
#             N_occluding, N_occluded, _, _ = \
#                 m_detector.get_occluding_and_occluded_points(current_pt_g_to_test, historical_di)
#             print(f"  Point-Level Result: {len(N_occluding)} occluding points, {len(N_occluded)} occluded points found in Historical DI.")

#             # --- Visualization for this Test Point ---
#             plot = k3d.plot(name=f"Occlusion Test - Point {i+1}", grid_visible=True, camera_auto_fit=False)
            
#             # A. Plot points from the Historical DI (context)
#             hist_di_all_points_global = []
#             for v_pix in range(historical_di.num_pixels_v):
#                 for h_pix in range(historical_di.num_pixels_h):
#                     px = historical_di.pixels[v_pix, h_pix]
#                     if px and px['points']:
#                         for pt_info in px['points']: hist_di_all_points_global.append(pt_info['global_pt'])
#             if hist_di_all_points_global:
#                 plot += k3d.points(np.array(hist_di_all_points_global), point_size=0.05, color=0x808080, name="Historical DI Pts (All)")

#             # B. Plot points from the Latest DI (context)
#             latest_di_all_points_global = []
#             for v_pix in range(latest_di.num_pixels_v):
#                 for h_pix in range(latest_di.num_pixels_h):
#                     px = latest_di.pixels[v_pix, h_pix]
#                     if px and px['points']:
#                         for pt_info in px['points']: latest_di_all_points_global.append(pt_info['global_pt'])
#             if latest_di_all_points_global:
#                  plot += k3d.points(np.array(latest_di_all_points_global), point_size=0.05, color=0xADD8E6, name="Latest DI Pts (All)") # Light blue


#             # C. Highlight the Current Test Point
#             test_pt_color = 0xff00ff # Magenta default
#             if pixel_res == OcclusionResult.OCCLUDED_BY_IMAGE: test_pt_color = 0xff0000 # Red
#             elif pixel_res == OcclusionResult.OCCLUDING_IMAGE: test_pt_color = 0x00ff00 # Green
#             elif pixel_res == OcclusionResult.EMPTY_IN_IMAGE: test_pt_color = 0xffff00 # Yellow
#             plot += k3d.points(current_pt_g_to_test.reshape(1,3), point_size=0.25, color=test_pt_color, name="Current Test Point")

#             # D. Highlight the pixel region in Historical DI that was checked
#             if pix_idx_hist:
#                 v_hist_proj, h_hist_proj = pix_idx_hist
#                 region_points_global = []
#                 for dv in range(-m_detector.neighbor_search_pixels_v, m_detector.neighbor_search_pixels_v + 1):
#                     for dh in range(-m_detector.neighbor_search_pixels_h, m_detector.neighbor_search_pixels_h + 1):
#                         pv = v_hist_proj + dv
#                         ph = h_hist_proj + dh
#                         if 0 <= pv < historical_di.num_pixels_v and 0 <= ph < historical_di.num_pixels_h:
#                             px_data = historical_di.pixels[pv, ph]
#                             if px_data and px_data['points']:
#                                 for pt_info in px_data['points']: region_points_global.append(pt_info['global_pt'])
#                 if region_points_global:
#                     plot += k3d.points(np.array(region_points_global), point_size=0.1, color=0x000000, name="Hist. DI Checked Region Pts") # Black


#             # E. Highlight N_occluding and N_occluded points from Point-Level Check
#             if N_occluding:
#                 occluding_pts_coords = np.array([p['global_pt'] for p in N_occluding])
#                 plot += k3d.points(occluding_pts_coords, point_size=0.18, color=0xff8c00, name="N_occluding (from Hist DI)") # Dark Orange
#                 for occ_pt_g in occluding_pts_coords: # Lines to test point
#                     plot += k3d.line(np.vstack([current_pt_g_to_test, occ_pt_g]), shader='simple', width=0.02, color=0xff8c00)
            
#             if N_occluded:
#                 occluded_pts_coords = np.array([p['global_pt'] for p in N_occluded])
#                 plot += k3d.points(occluded_pts_coords, point_size=0.18, color=0x00ced1, name="N_occluded (by Test Pt in Hist DI)") # Dark Turquoise
#                 for occ_pt_g in occluded_pts_coords: # Lines from test point
#                      plot += k3d.line(np.vstack([current_pt_g_to_test, occ_pt_g]), shader='simple', width=0.02, color=0x00ced1)

#             # Set camera: look from latest_di's sensor position towards the test point
#             cam_eye = latest_di.image_pose_global[:3, 3] # LiDAR sensor origin of latest_di
#             cam_look_at = current_pt_g_to_test
#             # Simple up vector, assuming Z is generally up in global frame
#             # More robustly, use the Z-axis of latest_di.image_pose_global
#             cam_up = latest_di.image_pose_global[:3, 2] if np.linalg.norm(latest_di.image_pose_global[:3, 2]) > 0.1 else [0,0,1]

#             # Adjust eye position to be slightly behind and above the sensor origin for a better view
#             view_direction = cam_look_at - cam_eye
#             view_direction_norm = view_direction / (np.linalg.norm(view_direction) + 1e-6)
            
#             # Offset the camera eye position slightly back from the sensor and a bit to the side/up
#             # This is an art to get a good default view.
#             # Using the DI's x-axis (right) and z-axis (up) for offset
#             offset_backward = latest_di.image_pose_global[:3, 0] * -3 # 3m behind along DI's x-axis (forward)
#             offset_sideways = latest_di.image_pose_global[:3, 1] * 1 # 1m to the DI's y-axis (left)
#             offset_up = latest_di.image_pose_global[:3, 2] * 1 # 1m up along DI's z-axis
            
#             # If test point is far, move camera further back
#             dist_to_point = np.linalg.norm(view_direction)
#             cam_eye_adjusted = cam_eye - view_direction_norm * max(5, dist_to_point * 0.3) + offset_up # Pull back based on distance
            
#             plot.camera = [
#                 cam_eye_adjusted[0], cam_eye_adjusted[1], cam_eye_adjusted[2],
#                 cam_look_at[0], cam_look_at[1], cam_look_at[2],
#                 cam_up[0], cam_up[1], cam_up[2]
#             ]
#             plot.display()

In [ ]:
import os
import numpy as np
import yaml
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MatplotlibPolygon
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box # Only Box is needed from here for annotations
from pyquaternion import Quaternion
import cv2

# --- Configuration for Video Generation ---
VIDEO_SCENE_INDEX = 1
VIDEO_START_SWEEP_NUM_LIDAR = 0 # Process from the beginning of the scene

OUTPUT_VIDEO_FILENAME = f'mdetector_scene_{VIDEO_SCENE_INDEX}_dynamic_pts_v2.mp4'
VIDEO_FPS = 20

BEV_RANGE_METERS = 50
POINT_SIZE_ALL_LIDAR = 0.2
POINT_SIZE_DYNAMIC = 2.0
EGO_VEHICLE_COLOR = 'blue'
LIDAR_POINT_COLOR = 'lightgrey'
DYNAMIC_POINT_COLOR = 'green'
BOX_EDGE_COLOR = 'red'
BOX_LINE_WIDTH = 1
HISTORICAL_DI_LAG_SECONDS = 0.5

# --- Helper function to draw a 2D BEV box (same as before) ---
def draw_bev_box_on_ax(ax, box: Box, color='red', linewidth=1):
    # ... (implementation from before is fine) ...
    corners_3d = box.corners()
    points_for_polygon = np.transpose(corners_3d[:2, [0, 1, 2, 3]])
    ax.add_patch(MatplotlibPolygon(points_for_polygon, closed=True, fill=False, edgecolor=color, linewidth=linewidth))
    center_bottom_face = np.mean(corners_3d[:, [0,1,2,3]], axis=1)
    orientation_vec = box.orientation.rotate(np.array([box.wlh[0] / 2.0, 0, 0]))
    arrow_start = center_bottom_face[:2]
    arrow_end = center_bottom_face[:2] + orientation_vec[:2]
    ax.arrow(arrow_start[0], arrow_start[1],
             arrow_end[0] - arrow_start[0], arrow_end[1] - arrow_start[0],
             head_width=0.5, head_length=0.7, fc=color, ec=color, linewidth=linewidth*0.5)


# --- Helper: Matplotlib figure to OpenCV BGR image (same as before) ---
def mpl_fig_to_opencv_bgr(fig):
    # ... (implementation from before is fine) ...
    fig.canvas.draw()
    img_np_rgb = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img_np_rgb = img_np_rgb.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    img_np_bgr = cv2.cvtColor(img_np_rgb, cv2.COLOR_RGB2BGR)
    return img_np_bgr

# --- Helper: Transform points (N,3) using a 4x4 matrix ---
def transform_points_numpy(points_n3, transf_matrix_4x4):
    """Transforms (N,3) points using a 4x4 transformation matrix."""
    points_n4_homogeneous = np.hstack((points_n3, np.ones((points_n3.shape[0], 1)))) # (N,4)
    points_transformed_n4 = points_n4_homogeneous @ transf_matrix_4x4.T # (N,4)
    return points_transformed_n4[:, :3] # Back to (N,3)

# --- Main Video Generation Script ---
if __name__ == '__main__':
    config_file_path = '../config/m_detector_config.yaml'
    with open(config_file_path, 'r') as f:
        config_yaml = yaml.safe_load(f) # Renamed to avoid conflict with m_detector's config attribute

    print(f"Loading NuScenes {config_yaml['nuscenes']['version']} from {config_yaml['nuscenes']['dataroot']}...")
    nusc = NuScenes(version=config_yaml['nuscenes']['version'],
                    dataroot=config_yaml['nuscenes']['dataroot'],
                    verbose=False)

    print("Initializing MDetector...")
    m_detector = MDetector(config=config_yaml)

    my_scene = nusc.scene[VIDEO_SCENE_INDEX]
    print(f"Targeting scene: {my_scene['name']} (Token: {my_scene['token']})")

    first_sample_token = my_scene['first_sample_token']
    first_sample = nusc.get('sample', first_sample_token)
    current_lidar_data_token = first_sample['data'][config_yaml['nuscenes']['lidar_sensor_name']]
    
    sweep_counter_for_start = 0
    if VIDEO_START_SWEEP_NUM_LIDAR > 0:
        print(f"Skipping to LiDAR sweep approx #{VIDEO_START_SWEEP_NUM_LIDAR}...")
        while current_lidar_data_token and sweep_counter_for_start < VIDEO_START_SWEEP_NUM_LIDAR:
            lidar_data_rec = nusc.get('sample_data', current_lidar_data_token)
            current_lidar_data_token = lidar_data_rec['next']
            sweep_counter_for_start += 1
            if not current_lidar_data_token:
                print(f"Error: Reached end of scene before reaching start sweep {VIDEO_START_SWEEP_NUM_LIDAR}")
                exit()
        print(f"Starting video generation from LiDAR sweep approx #{VIDEO_START_SWEEP_NUM_LIDAR}")


    fig_init, ax_init = plt.subplots(figsize=(10, 10))
    frame_bgr = mpl_fig_to_opencv_bgr(fig_init)
    frame_height, frame_width, _ = frame_bgr.shape
    plt.close(fig_init)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(OUTPUT_VIDEO_FILENAME, fourcc, VIDEO_FPS, (frame_width, frame_height))
    print(f"Output video: {OUTPUT_VIDEO_FILENAME} ({frame_width}x{frame_height} @ {VIDEO_FPS} FPS)")

    fig_main, ax_main = plt.subplots(figsize=(10, 10))
    processed_video_frames = 0
    current_dynamic_points_global_to_plot = []

    while current_lidar_data_token: # Process all available sweeps from the start token
        lidar_data = nusc.get('sample_data', current_lidar_data_token)
        ego_pose_data = nusc.get('ego_pose', lidar_data['ego_pose_token'])
        sample_for_annotations = nusc.get('sample', lidar_data['sample_token'])

        print(f"Processing video frame {processed_video_frames + 1}, LiDAR Timestamp: {lidar_data['timestamp']/1e6:.2f}s")

        # Get current sweep data (points_lidar_sensor_frame is (N,3) in sensor frame)
        points_lidar_sensor_frame, T_global_lidar, lidar_timestamp = \
            get_lidar_sweep_data(nusc, current_lidar_data_token) # This should be your working one

        # --- M-Detector Processing ---
        current_di = m_detector.add_sweep_and_create_depth_image(
            points_lidar_sensor_frame, T_global_lidar, lidar_timestamp
        )
        
        newly_detected_dynamic_points_global = []
        if m_detector.is_ready_for_processing() and current_di:
            historical_di = None
            if len(m_detector.depth_image_library) >=2:
                historical_di = m_detector.depth_image_library.get_image_by_index(-2) # Get DI before current_di

            if historical_di and historical_di.timestamp < current_di.timestamp:
                for v_idx in range(current_di.num_pixels_v):
                    for h_idx in range(current_di.num_pixels_h):
                        pixel_content = current_di.pixels[v_idx, h_idx]
                        if pixel_content and pixel_content['points']:
                            for pt_info in pixel_content['points']:
                                global_pt_curr = pt_info['global_pt']
                                pixel_res, _, _ = m_detector.check_occlusion_pixel_level(global_pt_curr, historical_di)
                                if pixel_res == OcclusionResult.OCCLUDING_IMAGE:
                                    newly_detected_dynamic_points_global.append(global_pt_curr)
                current_dynamic_points_global_to_plot = newly_detected_dynamic_points_global
        
        # --- Plotting current 20Hz LiDAR sweep ---
        ax_main.clear()

        # Transform current LiDAR points (sensor frame) to global frame for BEV plot
        # points_lidar_sensor_frame is (N,3)
        # T_global_lidar is (4,4)
        points_global_plot = transform_points_numpy(points_lidar_sensor_frame, T_global_lidar) # (N,3) in global

        ax_main.scatter(points_global_plot[:, 0], points_global_plot[:, 1], s=POINT_SIZE_ALL_LIDAR, color=LIDAR_POINT_COLOR, alpha=0.6)

        # Plot Ego Vehicle
        ego_translation_global = np.array(ego_pose_data['translation'])
        ego_rotation_global = Quaternion(ego_pose_data['rotation'])
        ax_main.plot(ego_translation_global[0], ego_translation_global[1], 'o', markersize=8, color=EGO_VEHICLE_COLOR, label='Ego Vehicle')
        ego_front_direction = ego_rotation_global.rotate(np.array([2.0, 0, 0]))
        ax_main.arrow(ego_translation_global[0], ego_translation_global[1],
                       ego_front_direction[0], ego_front_direction[1],
                       head_width=0.8, head_length=1.0, fc=EGO_VEHICLE_COLOR, ec=EGO_VEHICLE_COLOR)
        
        # Plot M-Detector's dynamic points
        if current_dynamic_points_global_to_plot:
            dynamic_pts_np = np.array(current_dynamic_points_global_to_plot) # Already global
            ax_main.scatter(dynamic_pts_np[:, 0], dynamic_pts_np[:, 1], s=POINT_SIZE_DYNAMIC, color=DYNAMIC_POINT_COLOR, label='MDet: OCCLUDING', zorder=5)

        # Plot settings
        ax_main.set_xlim(ego_translation_global[0] - BEV_RANGE_METERS, ego_translation_global[0] + BEV_RANGE_METERS)
        ax_main.set_ylim(ego_translation_global[1] - BEV_RANGE_METERS, ego_translation_global[1] + BEV_RANGE_METERS)
        ax_main.set_aspect('equal', adjustable='box')
        ax_main.set_xlabel("Global X (meters)")
        ax_main.set_ylabel("Global Y (meters)")
        title_str = (f"Scene: {my_scene['name']} - LiDAR Sweep Frame: {processed_video_frames}\n"
                     f"TS: {lidar_data['timestamp']/1e6:.2f}s - MDet Ready: {m_detector.is_ready_for_processing()} "
                     f"DI Lib: {len(m_detector.depth_image_library)}")
        if current_dynamic_points_global_to_plot: title_str += f" DynPts: {len(current_dynamic_points_global_to_plot)}"
        ax_main.set_title(title_str, fontsize=9)

        ax_main.grid(True, linestyle='--', alpha=0.5)
        if processed_video_frames == 0: ax_main.legend(loc='upper right', fontsize='small')

        video_frame_bgr = mpl_fig_to_opencv_bgr(fig_main)
        video_writer.write(video_frame_bgr)

        processed_video_frames += 1
        current_lidar_data_token = lidar_data['next']
        # if processed_video_frames >= VIDEO_NUM_LIDAR_SWEEPS_TO_PROCESS: # Uncomment to limit frames
        #     break

    plt.close(fig_main)
    video_writer.release()
    print(f"\nVideo generation complete. Saved {processed_video_frames} frames to {OUTPUT_VIDEO_FILENAME}")